# 06m. 파인튜닝 한 번에 돌리기 (큐): 모델 선택판

`finetuning_model.ipynb`를 설정마다 그대로 실행한다. **모델(`MODEL`)을 하이퍼파라미터로** 고르고, 결과는 모델 이름 폴더에 쌓인다. A·B, 방식(전술 / 기법 / 전술+기법), k = 1 ~ 16, 학습률을 **한 줄로 세워 차례로** 돌린다.

- 코드는 `finetuning_model.ipynb` 하나뿐이다. 이 노트북은 설정 칸의 값(`MODEL`, `VARIANT`, `EPOCHS`, `ENCODER_LR`, `OTHERS_LR`, `TAG`)만 바꿔서 실행한다. 설정 칸 원문이 바뀌어 값을 못 넣으면 멈춘다
- 실행된 노트북은 결과 폴더에 `executed.ipynb`로 남는다 (실패하면 `failed.ipynb`). `executed.ipynb`가 있는 설정은 건너뛰므로 중간에 끊겨도 다시 실행하면 이어서 한다
- 결과: `data/v2/{설정}/results/{모델 이름}/finetune_{설정}_lr{lr}_ep{에폭}{SUFFIX}/` (모델 이름 = `MODEL`의 `/` 뒤)
- **A / B**: A = 윈도우로 나눈 뒤 같은 내용 윈도우 병합, B = 윈도우 전에 이벤트 반복 압축

**Colab에서**
1. 런타임 유형: GPU
2. 데이터: 드라이브에 `ctgli_v2*.tar.xz`(`data/v2` 묶음)와 이 저장소의 `notebooks/`를 올리거나, `DATA_DIR`·`FINETUNE_NB`를 직접 지정
3. 칸을 차례로 실행. `BACKGROUND = True`면 3번 칸이 바로 끝나고 학습은 런타임 안의 별도 프로세스로 돈다. 4번 칸으로 진행 상황을 본다
4. 브라우저를 닫아도 학습을 이어가려면 Colab Pro+의 **백그라운드 실행**을 켠다 (아니면 탭을 닫으면 런타임이 끝난다)

설정 하나만 돌릴 때는 `06_finetune_single_model.ipynb`.

## 1. 설정

In [ ]:
MODEL = 'knowledgator/gliclass-edge-v3.0'   # 하이퍼파라미터: GLiClass 모델 (예: knowledgator/gliclass-modern-large-v3.0)
VARIANTS = ['A', 'B']                     # 데이터: A(윈도우 병합), B(이벤트 반복 압축)
MODES = ['both', 'tactic', 'technique']    # 이 순서대로. 전술+기법(both)을 먼저
KS = [1, 3, 5, 8, 16]
LRS = ['2e-5', '5e-5']
EPOCHS = 40
SUFFIX = '_fix'                            # 결과 폴더 이름 끝 (TAG = _lr{lr}_ep{EPOCHS}{SUFFIX})
SMOKE = False                              # True: 설정마다 2에폭·평가 일부 (끝까지 도는지 확인용)
BACKGROUND = True                          # True: 별도 프로세스로 돌리고 칸은 바로 끝남 (Colab 권장)

DATA_DIR = None                            # None이면 자동: 저장소 data/, /content/ctgli/data, 드라이브의 ctgli_v2*.tar.xz
FINETUNE_NB = None                         # None이면 자동: notebooks/finetuning.ipynb 등
WORK_DIR = None                            # 체크포인트·모델 (None이면 Colab /content/work, 로컬 data/finetune_work)
LOG = None                                 # None이면 WORK_DIR/queue_{모델 이름}.log

# 순서: 방식 → k → 데이터 → 학습률 (전술+기법을 모든 k에서 먼저 끝냄)
MODEL_NAME = MODEL.rstrip('/').split('/')[-1]
JOBS = [(f'{v}_{m}_k{k}', lr) for m in MODES for k in KS for v in VARIANTS for lr in LRS]
print(MODEL_NAME, '|', len(JOBS), '개:', JOBS[:4], '...')

## 2. 준비

[하는 일] 데이터와 `finetuning.ipynb`를 찾고, 노트북 실행 도구(`nbclient`)를 설치한다. GLiClass 설치는 `finetuning.ipynb`의 1번 칸이 한다.

[볼 것] `분할 폴더 = 30 개`, GPU 이름.

In [ ]:
import os, sys, glob, subprocess, tarfile, importlib.util
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def find_data():
    """data/v2가 있는 폴더 (DATA_DIR). 없으면 드라이브의 ctgli_v2*.tar.xz를 /content/ctgli/data에 푼다."""
    if DATA_DIR and Path(DATA_DIR, 'v2').exists():
        return str(DATA_DIR)
    here = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    for c in [here / 'data', Path('/content/ctgli/data')]:
        if (c / 'v2').exists():
            return str(c)
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
        for z in sorted(glob.glob('/content/drive/MyDrive/**/ctgli_v2*.tar.xz', recursive=True)):
            os.makedirs('/content/ctgli/data', exist_ok=True)
            tarfile.open(z).extractall('/content/ctgli/data', filter='data'); print('압축 풀기:', z)
        if Path('/content/ctgli/data/v2').exists():
            return '/content/ctgli/data'
    raise FileNotFoundError('data/v2를 못 찾음: DATA_DIR을 지정하거나 드라이브에 ctgli_v2*.tar.xz를 올릴 것')

def find_notebook():
    if FINETUNE_NB and Path(FINETUNE_NB).exists():
        return str(FINETUNE_NB)
    for c in [Path.cwd() / 'finetuning_model.ipynb', Path.cwd() / 'notebooks' / 'finetuning_model.ipynb',
              Path('/content/ctgli/notebooks/finetuning_model.ipynb'), Path('/content/ctgli/finetuning_model.ipynb')]:
        if c.exists():
            return str(c)
    raise FileNotFoundError('finetuning_model.ipynb를 못 찾음: FINETUNE_NB를 지정할 것')

DATA_DIR, FINETUNE_NB = find_data(), find_notebook()
WORK_DIR = WORK_DIR or ('/content/work' if IN_COLAB else os.path.join(DATA_DIR, 'finetune_work'))
need = [p for p, m in [('nbclient', 'nbclient'), ('nbformat', 'nbformat'), ('ipykernel', 'ipykernel')] if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need], check=True)
gpu = subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', shell=True, capture_output=True, text=True).stdout.strip()
print('DATA_DIR    =', DATA_DIR)
print('FINETUNE_NB =', FINETUNE_NB)
print('WORK_DIR    =', WORK_DIR)
print('GPU         =', gpu or '없음 (CPU로는 매우 느림)')
print('분할 폴더    =', len(glob.glob(f'{DATA_DIR}/v2/[AB]_*_k*/splits')), '개')

## 3. 실행

[하는 일] 큐 실행 스크립트(`finetune_runner.py`)를 쓰고 돌린다. 설정 하나가 끝날 때마다 로그에 `{설정} {학습률} done {초} s`가 남는다.

In [ ]:
RUNNER = r"""
import json, os, re, sys, time
import nbformat
from nbclient import NotebookClient

cfg = json.loads(sys.argv[1])
os.environ['DATA_DIR'], os.environ['WORK_DIR'] = cfg['data_dir'], cfg['work_dir']

def params(nb, variant, lr, epochs, tag, smoke):
    # finetuning.ipynb 설정 칸의 원문을 바꾼다. 원문이 달라졌으면 멈춘다 (다른 설정으로 조용히 도는 것을 막음)
    reps = [("MODEL = 'knowledgator/gliclass-edge-v3.0'", f"MODEL = '{cfg['model']}'"),
            ("VARIANT = 'B_tactic_k16'", f"VARIANT = '{variant}'"),
            ("EPOCHS = 10", f"EPOCHS = {epochs}"),
            ("TAG = ''", f"TAG = '{tag}'")]
    if smoke:
        reps.append(("SMOKE = False", "SMOKE = True"))
    cell = next(c for c in nb.cells if c.cell_type == 'code' and "VARIANT = 'B_tactic_k16'" in c.source)
    for a, b in reps:
        assert a in cell.source, f'finetuning.ipynb 설정 칸에 {a!r}가 없음'
        cell.source = cell.source.replace(a, b)
    cell.source = re.sub(r"ENCODER_LR = [0-9e.-]+", f"ENCODER_LR = {lr}", cell.source)
    cell.source = re.sub(r"OTHERS_LR = [0-9e.-]+", f"OTHERS_LR = {lr}", cell.source)

for variant, lr in cfg['jobs']:
    tag = f"_lr{lr}_ep{cfg['epochs']}{cfg['suffix']}"
    model_name = cfg['model'].rstrip('/').split('/')[-1]
    out = f"{cfg['data_dir']}/v2/{variant}/results/{model_name}/finetune_{variant}{tag}" + ('_smoke' if cfg['smoke'] else '')
    if os.path.exists(f'{out}/executed.ipynb'):
        print(variant, lr, 'skip (done)', flush=True); continue
    nb = nbformat.read(cfg['notebook'], 4)
    params(nb, variant, lr, cfg['epochs'], tag, cfg['smoke'])
    t = time.time()
    try:
        NotebookClient(nb, kernel_name='python3', timeout=6 * 3600,
                       resources={'metadata': {'path': os.path.dirname(cfg['notebook'])}}).execute()
        status = 'done'
    except Exception as e:
        status = 'ERROR ' + (str(e).strip().splitlines() or [''])[-1][:300]
    os.makedirs(out, exist_ok=True)
    nbformat.write(nb, f'{out}/executed.ipynb' if status == 'done' else f'{out}/failed.ipynb')
    for c in nb.cells:
        if c.cell_type == 'code':
            for o in c.outputs:
                for line in (o.get('text') or '').splitlines():
                    if line.startswith(('k=', '검사', '  학습', '  추론')):
                        print(variant, lr, line, flush=True)
    print(variant, lr, status, round(time.time() - t), 's', flush=True)
print('ALL FINISHED', flush=True)
"""
RUNNER_PY = os.path.join(WORK_DIR, 'finetune_runner_model.py')
os.makedirs(WORK_DIR, exist_ok=True)
open(RUNNER_PY, 'w').write(RUNNER)
import json
LOG = LOG or os.path.join(WORK_DIR, f'queue_{MODEL_NAME}.log')
cfg = json.dumps(dict(model=MODEL, jobs=JOBS, epochs=EPOCHS, suffix=SUFFIX, smoke=SMOKE, data_dir=DATA_DIR,
                      work_dir=WORK_DIR, notebook=os.path.abspath(FINETUNE_NB)))
if BACKGROUND:
    with open(LOG, 'a') as log:
        proc = subprocess.Popen([sys.executable, RUNNER_PY, cfg], stdout=log, stderr=subprocess.STDOUT,
                                start_new_session=True)
    print(f'백그라운드 시작 (pid {proc.pid}). 로그: {LOG}  → 4번 칸으로 진행 확인')
else:
    subprocess.run([sys.executable, RUNNER_PY, cfg], check=False)

## 4. 진행 상황

[하는 일] 로그에서 끝난 설정을 세고, 마지막 몇 줄을 보여준다. 몇 번이고 다시 실행해도 된다.

In [ ]:
import re, collections
LOG = LOG or os.path.join(WORK_DIR, f'queue_{MODEL_NAME}.log')
text = open(LOG).read() if os.path.exists(LOG) else ''
done = {tuple(m.groups()[:2]) for m in re.finditer(r'^(\S+) (\S+) (done|skip)', text, re.M)}
err = [l for l in text.splitlines() if ' ERROR' in l]
running = subprocess.run(['pgrep', '-f', 'finetune_runner_model.py'], capture_output=True, text=True).stdout.split()
print(f'끝남 {len(done & set(map(tuple, JOBS)))}/{len(JOBS)} | 오류 {len(err)} | 실행 중: {"예" if running else "아니오"}')
by_mode = collections.Counter(v.split('_')[1] for v, lr in done & set(map(tuple, JOBS)))
print('방식별:', dict(by_mode))
for line in [l for l in text.splitlines() if l.split(' ')[0][:2] in ('A_', 'B_')][-8:]:
    print(' ', line[:200])
for l in err[-3:]:
    print('오류:', l[:300])

## 5. 결과 모음

[하는 일] 이 모델(`MODEL_NAME`)의 끝난 설정 `runs.csv`를 모아 seed 평균 표를 만들고 `{DATA_DIR}/v2/summary_{모델 이름}{SUFFIX}.csv`로 저장한다. `*_base`는 파인튜닝 전 기준선.

- `seen_macro_f1`: 학습 라벨 F1
- `held_macro_ap`: 늘 숨긴 라벨 AP (k끼리 비교 가능한 제로샷)
- `heldk_macro_ap`: 그 k에서만 숨긴 라벨 AP
- `seen_separation`: 공격 판별력 (recall - FPR)
- `agnews_acc`: 원래 실력 (기준선 약 0.588)

In [ ]:
import pandas as pd

def summarize(data_dir, suffix):
    """결과 폴더마다 runs.csv의 seed 평균 (k = 0 기준선은 빼고)."""
    rows = []
    for f in sorted(glob.glob(f'{data_dir}/v2/*/results/{MODEL_NAME}/finetune_*{suffix}/runs.csv')):
        variant = f.split('/v2/')[1].split('/')[0]
        lr = f.split('_lr')[1].split('_ep')[0]
        d = pd.read_csv(f)
        ft = d[d.k != 0]
        if not len(ft):
            continue
        r = {'variant': variant, 'data': variant[0], 'mode': variant.split('_')[1], 'k': int(variant.split('_k')[1]),
             'lr': lr, 'seeds': len(ft)}
        for kind in ['tactic', 'technique']:
            p = f'{kind}_' if f'{kind}_seen_macro_f1' in d else ('' if variant.split('_')[1] == kind else None)
            if p is None:
                continue
            for m in ['seen_macro_f1', 'held_macro_ap', 'heldk_macro_ap', 'seen_separation']:
                if p + m in d:
                    r[f'{kind}_{m}'] = round(ft[p + m].mean(), 3)
                    r[f'{kind}_{m}_base'] = round(d.loc[d.k == 0, p + m].mean(), 3)
        r['agnews_acc'] = round(ft['agnews_acc'].mean(), 3)
        rows.append(r)
    return pd.DataFrame(rows).sort_values(['data', 'mode', 'k', 'lr']).reset_index(drop=True) if rows else pd.DataFrame()

table = summarize(DATA_DIR, SUFFIX)
if len(table):
    table.to_csv(f'{DATA_DIR}/v2/summary_{MODEL_NAME}{SUFFIX}.csv', index=False)
    print('저장:', f'{DATA_DIR}/v2/summary_{MODEL_NAME}{SUFFIX}.csv')
    cols = ['variant', 'lr', 'seeds'] + [c for c in table.columns if c.endswith(('seen_macro_f1', 'held_macro_ap')) ] + ['agnews_acc']
    display(table[cols])
else:
    print('아직 끝난 설정 없음')